In [29]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import f1_score


df = pd.read_csv('finaldataset_with_sets.csv')

def views_to_class(views):
    if views < 10000:
        return 0
    elif 10000 <= views < 100000:
        return 1
    elif 100000 <= views < 500000:
        return 2
    elif 500000 <= views < 1000000:
        return 3
    else:
        return 4

df['label'] = df['views'].apply(views_to_class)

df.to_csv('finaldataset_with_sets.csv', index=False)

# Split into train and validation sets
train_df = df[df['set'] == 'training']
val_df = df[df['set'] == 'validation']


### Loading and Preparing data

In [797]:
# Load processed features 
def load_features(video_id, feature_dir='processedfeatures_final'):
    return np.load(f'{feature_dir}/{video_id}_corrected_features.npy')

# Load features for training and validation sets
train_features = np.array([load_features(video_id) for video_id in train_df['video_id']])
val_features = np.array([load_features(video_id) for video_id in val_df['video_id']])

# Convert labels to tensors
train_labels = torch.tensor(train_df['label'].values, dtype=torch.long)
val_labels = torch.tensor(val_df['label'].values, dtype=torch.long)

# Convert features to tensors
train_features = torch.tensor(train_features, dtype=torch.float32)
val_features = torch.tensor(val_features, dtype=torch.float32)


### Defining the Model

In [799]:
class CNNRNNModel(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, num_classes=5, num_layers=3):
        super(CNNRNNModel, self).__init__()
        self.rnn = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)  # out shape: [batch_size, seq_len, hidden_dim]
        out = self.fc(out)  # out shape: [batch_size, seq_len, num_classes]
        out = torch.mean(out, dim=1)  # Average across the sequence length, final shape: [batch_size, num_classes]
        return out


In [802]:
model = CNNRNNModel(input_dim=512, hidden_dim=128, num_classes=5, num_layers=3)  # Adjust as needed

### Training

In [61]:
# Model, criterion, and optimizer
model = CNNRNNModel(input_dim=512, hidden_dim=128, num_classes=num_classes, num_layers=3)  
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Early stopping criteria
patience = 3
best_val_f1 = 0
patience_counter = 0

# Training loop
num_epochs = 20
batch_size = 5

for epoch in range(num_epochs):
    model.train()
    permutation = torch.randperm(train_features.size(0))
    
    epoch_loss = 0
    epoch_preds = []
    epoch_labels = []
    
    for i in range(0, train_features.size(0), batch_size):
        indices = permutation[i:i + batch_size]
        batch_x, batch_y = train_features[indices], train_labels[indices]

        optimizer.zero_grad()
        outputs = model(batch_x)
        
        # Flatten the outputs and calculate loss
        outputs = outputs.view(-1, num_classes) 
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        epoch_labels.extend(batch_y.cpu().numpy())
    
    # Calculate metrics for the epoch
    train_f1 = f1_score(epoch_labels, epoch_preds, average='weighted')
    train_accuracy = np.mean(np.array(epoch_preds) == np.array(epoch_labels))
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(val_features)
        val_outputs = val_outputs.view(-1, num_classes)
        val_preds = val_outputs.argmax(dim=1).cpu().numpy()
        
        val_f1 = f1_score(val_labels.cpu().numpy(), val_preds, average='weighted')
        val_accuracy = np.mean(val_preds == val_labels.cpu().numpy())
        
    print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {epoch_loss / len(train_features):.4f}, '
          f'Train Accuracy: {train_accuracy:.4f}, Train F1: {train_f1:.4f}, '
          f'Val Loss: {criterion(val_outputs, val_labels).item():.4f}, '
          f'Val Accuracy: {val_accuracy:.4f}, Val F1: {val_f1:.4f}')
    
    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print("Early stopping triggered")
        break

# Final evaluation on the validation set
print(f'Final Validation Accuracy: {val_accuracy:.4f}')
print(f'Final Validation F1 Score: {best_val_f1:.4f}')

Epoch 1/20, Train Loss: 0.2573, Train Accuracy: 0.4850, Train F1: 0.3633, Val Loss: 1.3056, Val Accuracy: 0.5400, Val F1: 0.3857
Epoch 2/20, Train Loss: 0.2458, Train Accuracy: 0.5125, Train F1: 0.3985, Val Loss: 1.2248, Val Accuracy: 0.5400, Val F1: 0.3857
Epoch 3/20, Train Loss: 0.2445, Train Accuracy: 0.5150, Train F1: 0.3958, Val Loss: 1.2725, Val Accuracy: 0.5500, Val F1: 0.3903
Epoch 4/20, Train Loss: 0.2410, Train Accuracy: 0.5225, Train F1: 0.4126, Val Loss: 1.2447, Val Accuracy: 0.5200, Val F1: 0.3813
Epoch 5/20, Train Loss: 0.2375, Train Accuracy: 0.5625, Train F1: 0.4616, Val Loss: 1.3225, Val Accuracy: 0.5200, Val F1: 0.3788
Epoch 6/20, Train Loss: 0.2300, Train Accuracy: 0.5575, Train F1: 0.4744, Val Loss: 1.3377, Val Accuracy: 0.5100, Val F1: 0.4038
Epoch 7/20, Train Loss: 0.2308, Train Accuracy: 0.5500, Train F1: 0.4830, Val Loss: 1.2780, Val Accuracy: 0.5100, Val F1: 0.3715
Epoch 8/20, Train Loss: 0.2093, Train Accuracy: 0.5975, Train F1: 0.5366, Val Loss: 1.4027, Val A